In [16]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [17]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1)

        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)

        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(in_features=16 * 5 * 5, out_features=120)

        self.fc2 = nn.Linear(in_features=120, out_features=84)

        self.fc3 = nn.Linear(in_features=84, out_features=10)

    def forward(self, x):

        x = self.pool1(torch.tanh(self.conv1(x)))
        x = self.pool2(torch.tanh(self.conv2(x)))

        x = x.view(-1, 16 * 5 * 5)

        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)

        return x

In [18]:
model = LeNet5()

In [19]:
loss_fn =  nn.CrossEntropyLoss()
optimiser =  torch.optim.Adam(model.parameters(),lr=0.001)

In [20]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set,batch_size=1,shuffle=True)
test_loader = DataLoader(test_set,batch_size=1,shuffle=True)

In [21]:
print(model.parameters)
print([p.numel() for p in model.parameters()])
params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {params:,}")

<bound method Module.parameters of LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)>
[150, 6, 2400, 16, 48000, 120, 10080, 84, 840, 10]
Total Parameters: 61,706


In [22]:
from ptflops import get_model_complexity_info
macs, params = get_model_complexity_info(model, (1, 32, 32), as_strings=True, print_per_layer_stat=False)
print(f"Computational complexity: {macs}")
print(params)

Computational complexity: 435.65 KMac
61.71 k


In [23]:
macs, params = get_model_complexity_info(model, (1, 32, 32), as_strings=True, print_per_layer_stat=True)

LeNet5(
  61.71 k, 100.000% Params, 429.34 KMac, 98.553% MACs, 
  (conv1): Conv2d(156, 0.253% Params, 122.3 KMac, 28.074% MACs, 1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): AvgPool2d(0, 0.000% Params, 4.7 KMac, 1.080% MACs, kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(2.42 k, 3.915% Params, 241.6 KMac, 55.458% MACs, 6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(0, 0.000% Params, 1.6 KMac, 0.367% MACs, kernel_size=2, stride=2, padding=0)
  (fc1): Linear(48.12 k, 77.983% Params, 48.12 KMac, 11.046% MACs, in_features=400, out_features=120, bias=True)
  (fc2): Linear(10.16 k, 16.472% Params, 10.16 KMac, 2.333% MACs, in_features=120, out_features=84, bias=True)
  (fc3): Linear(850, 1.377% Params, 850.0 Mac, 0.195% MACs, in_features=84, out_features=10, bias=True)
)


In [ ]:
import tqdm
import time
from torch.utils.tensorboard import SummaryWriter
LeNet5_Metrics = SummaryWriter()

latency_per_image = dict()
latency_i = 0


for image,label in tqdm(train_loader):

    start_time = time.time()

    optimiser.zero_grad()

    y_pred =  model(image)

    loss = loss_fn(y_pred,label)

    loss.backward()

    optimiser.step()

    end_time = time.time()

    LeNet5_Metrics.add_scalar("Loss/train", loss.item(), global_step)
    LeNet5_Metrics.add_scalar("Accuracy/train", acc, global_step)

LeNet5_Metrics.close()